In [3]:
# RESTART RUNTIME AFTER THIS
!pip install pyspark==3.5.3 passim -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.3/317.3 MB 3.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.0.2 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.3 which is incompatible.


In [1]:
import pyspark
print(pyspark.__version__)  # must be 3.5.3

3.5.3


In [2]:
import json, sys, shutil
from pathlib import Path
import pandas as pd

from google.colab import drive
drive.mount('/content/drive')

DRIVE = Path("/content/drive/MyDrive/bullinger_passim")
assert DRIVE.exists(), f"Drive folder not found: {DRIVE}"

Mounted at /content/drive


In [3]:
def load_chunks(path):
    with open(path) as f:
        data = json.load(f)
    chunks = data.get('chunks', data) if isinstance(data, dict) else data
    print(f"  {Path(path).name}: {len(chunks):,}")
    return chunks

def write_jsonl(chunks, series, path):
    with open(path, 'w') as f:
        for c in chunks:
            f.write(json.dumps({'id': c['chunk_id'], 'text': c['text'], 'series': series}) + '\n')

print("Loaded chunks:")
bdc_chunks     = load_chunks(DRIVE / "VD_bdc_chunks.json")
psc_chunks     = load_chunks(DRIVE / "VD_psc_chunks.json")
vulgate_chunks = load_chunks(DRIVE / "vulgate_chunks.json")

!mkdir -p /content/input_bdc /content/input_psc
write_jsonl(vulgate_chunks, 0, "/content/input_bdc/vulgate.jsonl")
write_jsonl(bdc_chunks,     1, "/content/input_bdc/bdc.jsonl")
write_jsonl(vulgate_chunks, 0, "/content/input_psc/vulgate.jsonl")
write_jsonl(psc_chunks,     1, "/content/input_psc/psc.jsonl")

Loaded chunks:
  VD_bdc_chunks.json: 19,466
  VD_psc_chunks.json: 46,408
  vulgate_chunks.json: 35,809


In [4]:
!pip install git+https://github.com/dasmiq/passim.git -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 9.0 MB/s eta 0:00:00


In [5]:
# Parameter grid — each row is one PASSIM run
# (label, n, min_match, gap, min_align)
PARAM_GRID = [
    ("n3_mm2_g50_a5",   3, 2,  50,  5),  # trigrams loose
    ("n4_mm2_g50_a5",  4, 2, 50,  5),   # 4-grams, loose
    ("n4_mm3_g50_a8",  4, 3, 50,  8),   # 4-grams, strict
    ("n5_mm2_g50_a5",   5, 2,  50,  5),  # 5-grams loose
    ("n5_mm3_g50_a8",  5, 3,  50, 8),  # 5-grams, strict
]

In [6]:
# Write lemmatized JSONL
def write_jsonl_lemma(chunks, series, path):
    with open(path, 'w') as f:
        for c in chunks:
            lemmas = c.get('lemmatized', [])
            # Join lemma list to string, skip punctuation tokens
            text = ' '.join(
                t for t in lemmas
                if t not in {',', '.', ':', ';', '!', '?', '"', "'", '-', '(', ')'}
            )
            if not text.strip():
                continue
            f.write(json.dumps({
                'id':     c['chunk_id'],
                'text':   text,
                'series': series
            }) + '\n')

In [7]:
!mkdir -p /content/input_bdc_lemma /content/input_psc_lemma

In [8]:
write_jsonl_lemma(vulgate_chunks, 0, "/content/input_bdc_lemma/vulgate.jsonl")
write_jsonl_lemma(bdc_chunks,     1, "/content/input_bdc_lemma/bdc.jsonl")
print("Lemmatized JSONL ready.")

Lemmatized JSONL ready.


In [9]:
from passim import seriatim
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.getOrCreate()

#  uid->chunk_id mappign
bdc_uid_map = (
    spark.read.json("/content/input_bdc/bdc.jsonl")
    .withColumn('uid2', F.xxhash64('id'))
    .select(F.col('uid2'), F.col('id').alias('chunk_id'))
)
psc_uid_map = (
    spark.read.json("/content/input_psc/psc.jsonl")
    .withColumn('uid2', F.xxhash64('id'))
    .select(F.col('uid2'), F.col('id').alias('chunk_id'))
)

def run_passim(input_dir, output_dir, n, min_match, gap, min_align):
    if Path(output_dir).exists():
        shutil.rmtree(output_dir)
    sys.argv = [
        "passim", input_dir, output_dir,
        f"--n={n}", f"--min-match={min_match}",
        f"--gap={gap}", f"--min-align={min_align}",
        "--pcopy=0.8",
        "--fields", "series",
        "--filterpairs", "series != series2",
        "--to-extents",
    ]
    seriatim.main(sys.argv[1:])

def extract_scores(extents_dir, input_jsonl, label):
    from pyspark.sql import SparkSession
    from pyspark.sql import functions as F

    # Recreate session
    spark = SparkSession.builder.getOrCreate()

    p = Path(extents_dir)
    if not p.exists():
        print(f"  {label}: no extents dir found")
        return pd.DataFrame(columns=['chunk_id', 'max_alignment_size'])

    extents = (
        spark.read.parquet(str(p))
        .filter((F.col('series') == 0) & (F.col('series2') == 1))
        .withColumn('size', F.col('end') - F.col('begin'))
    )
    if extents.count() == 0:
        print(f"  {label}: 0 extents")
        return pd.DataFrame(columns=['chunk_id', 'max_alignment_size'])

    # Rebuild uid map with fresh session
    uid_map = (
        spark.read.json(input_jsonl)
        .withColumn('uid2', F.xxhash64('id'))
        .select(F.col('uid2'), F.col('id').alias('chunk_id'))
    )

    return (
        extents.join(uid_map, on='uid2', how='left')
        .groupBy('chunk_id')
        .agg(F.max('size').alias('max_alignment_size'))
        .toPandas()
        .dropna(subset=['chunk_id'])
    )

In [10]:
out_drive = DRIVE / "passim_output"
out_drive.mkdir(exist_ok=True)

In [18]:
summary = []

for label, n, min_match, gap, min_align in PARAM_GRID:
    print(f"\n{'='*55}")
    print(f"{label}  (n={n} min_match={min_match} gap={gap} min_align={min_align})")

    print("  BDC...")
    run_passim("/content/input_bdc", "/content/out_bdc", n, min_match, gap, min_align)
    bdc_scores = extract_scores("/content/out_bdc/tmp/extents.parquet/",
                                "/content/input_bdc/bdc.jsonl", "BDC")
    print(f"  BDC: {len(bdc_scores):,} chunks matched")

    #print("  PSC...")
    #run_passim("/content/input_psc", "/content/out_psc", n, min_match, gap, min_align)
    #psc_scores = extract_scores("/content/out_psc/tmp/extents.parquet/",
    #                            "/content/input_psc/psc.jsonl", "PSC")
    #print(f"  PSC: {len(psc_scores):,} chunks matched")

    bdc_scores.to_csv(str(out_drive / f"scores_bdc_{label}.csv"), index=False)
    #psc_scores.to_csv(str(out_drive / f"scores_psc_{label}.csv"), index=False)
    print(f"  Saved.")

    summary.append({
        'label': label, 'n': n, 'min_match': min_match,
        'gap': gap, 'min_align': min_align,
        'bdc_matched': len(bdc_scores),
        #'psc_matched': len(psc_scores),
    })

df_summary = pd.DataFrame(summary)
print("\nGrid complete:")
print(df_summary.to_string(index=False))
df_summary.to_csv(str(out_drive / "run_summary.csv"), index=False)


n4_mm3_g50_a5  (n=4 min_match=3 gap=50 min_align=5)
  BDC...
Namespace(id='id', text='text', locs='locs', pages='pages', minDF=2, maxDF=100, min_match=3, n=4, floating_ngrams=False, complete_lines=False, gap=50, max_offset=20, beam=20, pcopy=0.8, min_align=5, src_overlap=0.9, dst_overlap=0.5, fields=['series'], filterpairs='series != series2', all_pairs=False, pairwise=False, docwise=False, refpref=False, linewise=False, to_index=False, to_pairs=False, to_extents=True, link_model=None, link_features=None, log_level='WARN', input_format='json', output_format='json', inputPath='/content/input_bdc', outputPath='/content/out_bdc')
  BDC: 176 chunks matched
  Saved.

Grid complete:
        label  n  min_match  gap  min_align  bdc_matched
n4_mm3_g50_a5  4          3   50          5          176


In [12]:
# Run PASSIM on lemmatized text

summary = []

for label, n, min_match, gap, min_align in PARAM_GRID:
    print(f"\n{'='*55}")
    print(f"{label}  (n={n} min_match={min_match} gap={gap} min_align={min_align})")

    print("  BDC lemma...")
    run_passim("/content/input_bdc_lemma", "/content/out_bdc_lemma",
           n, min_match, gap, min_align)
    bdc_lemma_scores = extract_scores(
    "/content/out_bdc_lemma/tmp/extents.parquet/",
    "/content/input_bdc_lemma/bdc.jsonl", "BDC-lemma"
    )
    print(f"  BDC lemma: {len(bdc_lemma_scores):,} chunks matched")

    bdc_lemma_scores.to_csv(
    str(out_drive / f"scores_bdc_lemma_{label}.csv"), index=False
    )
    print("  Saved.")

    summary.append({
        'label': label, 'n': n, 'min_match': min_match,
        'gap': gap, 'min_align': min_align,
        'bdc_matched': len(bdc_lemma_scores),
    })

df_summary = pd.DataFrame(summary)
print("\nGrid complete:")
print(df_summary.to_string(index=False))
df_summary.to_csv(str(out_drive / "run_summary.csv"), index=False)


n3_mm2_g50_a5  (n=3 min_match=2 gap=50 min_align=5)
  BDC lemma...
Namespace(id='id', text='text', locs='locs', pages='pages', minDF=2, maxDF=100, min_match=2, n=3, floating_ngrams=False, complete_lines=False, gap=50, max_offset=20, beam=20, pcopy=0.8, min_align=5, src_overlap=0.9, dst_overlap=0.5, fields=['series'], filterpairs='series != series2', all_pairs=False, pairwise=False, docwise=False, refpref=False, linewise=False, to_index=False, to_pairs=False, to_extents=True, link_model=None, link_features=None, log_level='WARN', input_format='json', output_format='json', inputPath='/content/input_bdc_lemma', outputPath='/content/out_bdc_lemma')
  BDC lemma: 16 chunks matched
  Saved.

n4_mm2_g50_a5  (n=4 min_match=2 gap=50 min_align=5)
  BDC lemma...
Namespace(id='id', text='text', locs='locs', pages='pages', minDF=2, maxDF=100, min_match=2, n=4, floating_ngrams=False, complete_lines=False, gap=50, max_offset=20, beam=20, pcopy=0.8, min_align=5, src_overlap=0.9, dst_overlap=0.5, field